<a href="https://colab.research.google.com/github/Physalis-Alkekengi/ASIGROMACS/blob/main/PostCharmmGromacs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title GROMACS 2023.3 Compilation Pipeline (CUDA-Enabled)
# Note: Compilation time is approximately 45 minutes.

import os
import subprocess
import sys
import shutil
from IPython.display import display, Javascript

print("Initializing GROMACS build environment...")
print("Target Architecture: NVIDIA CUDA (GROMACS 2023.3)")

# 1. DEPENDENCY RESOLUTION
print("\nResolving build dependencies (APT)...")
subprocess.run("apt-get update -y > /dev/null", shell=True)
subprocess.run("apt-get install -y cmake build-essential libfftw3-dev > /dev/null", shell=True)

# 2. SOURCE ACQUISITION
if not os.path.exists("gromacs-2023.3.tar.gz"):
    print("Retrieving source code...")
    subprocess.run("wget https://ftp.gromacs.org/gromacs/gromacs-2023.3.tar.gz > /dev/null", shell=True)
    subprocess.run("tar xzf gromacs-2023.3.tar.gz > /dev/null", shell=True)

# 3. BUILD CONFIGURATION (CMAKE)
src_dir = os.path.abspath("gromacs-2023.3")
build_dir = os.path.join(src_dir, "build")

if not os.path.exists(build_dir):
    os.makedirs(build_dir)

os.chdir(build_dir)
print("\nConfiguring build system (CMake)...")

# Configuration flags:
# -DGMX_BUILD_OWN_FFTW=ON: Internal FFTW build ensures compatibility
# -DGMX_GPU=CUDA: Enables NVIDIA GPU acceleration
# -DREGRESSIONTEST_DOWNLOAD=OFF: Skips test data download to save bandwidth
cmake_cmd = [
    "cmake", "..",
    "-DGMX_BUILD_OWN_FFTW=ON",
    "-DREGRESSIONTEST_DOWNLOAD=OFF",
    "-DGMX_GPU=CUDA",
    "-DCMAKE_INSTALL_PREFIX=/usr/local/gromacs"
]

subprocess.run(cmake_cmd, check=True)

# 4. COMPILATION
print("\nInitiating compilation sequence...")
print("(Estimated duration: 30-45 minutes)")

# Parallel compilation restricted to 2 cores (-j 2) to prevent RAM exhaustion
try:
    subprocess.run("make -j 2", shell=True, check=True)
except subprocess.CalledProcessError:
    print("Parallel compilation failed. Retrying with single thread...")
    subprocess.run("make", shell=True, check=True)

# 5. INSTALLATION AND LINKING
print("\nInstalling binaries...")
subprocess.run("make install", shell=True, check=True)
subprocess.run("ln -s /usr/local/gromacs/bin/gmx /usr/bin/gmx_gpu", shell=True)

print("\nInstallation complete. Binary linked to /usr/bin/gmx_gpu.")
os.chdir("/content") # Return to root

# 6. SESSION PERSISTENCE MECHANISM
def keep_alive():
    # Injects JavaScript to simulate interaction, preventing session timeout during long compiles
    display(Javascript('''
        function ClickConnect(){
            console.log("KeepAlive: Clicking colab button");
            document.querySelector("colab-connect-button").click()
        }
        setInterval(ClickConnect, 60000)
    '''))
    print("Session persistence active.")

keep_alive()

In [ ]:
# @title Automated GROMACS GPU Simulation Pipeline
import os
import subprocess
import glob
import shutil
import sys

print("Initializing GPU simulation pipeline...")

# ==============================================================================
# 1. GROMACS BINARY DETECTION
# ==============================================================================
# Initialize variable for absolute path to the executable
gmx_executable = None

print("Scanning system for GROMACS binary...")

# 1. Extraction of custom binary archive (if present)
if os.path.exists("MY_GROMACS_GPU.zip") and not os.path.exists("/content/bin/gmx"):
    print("Extracting GROMACS binaries from archive...")
    # Unzip with quiet flag (-q) and no-overwrite (-n)
    subprocess.run("unzip -n -q MY_GROMACS_GPU.zip -d /content", shell=True)

# 2. Binary Location Resolution
# Recursive search for 'gmx' executable within the content directory
try:
    found_path = subprocess.check_output("find /content -name 'gmx' -type f | head -n 1", shell=True).decode().strip()
except:
    found_path = ""

if found_path and os.path.exists(found_path):
    # Store absolute path
    gmx_executable = os.path.abspath(found_path)
    print(f"Binary identified at: {gmx_executable}")

    # Enforce execution permissions
    subprocess.run(f"chmod +x '{gmx_executable}'", shell=True)

    # Dynamic Linker Configuration (LD_LIBRARY_PATH)
    # Ensures CUDA/OpenCL libraries are accessible during runtime
    bin_dir = os.path.dirname(gmx_executable)
    root_dir = os.path.dirname(bin_dir)
    lib_dir = os.path.join(root_dir, "lib")

    if os.path.exists(lib_dir):
        os.environ["LD_LIBRARY_PATH"] = f"{lib_dir}:{os.environ.get('LD_LIBRARY_PATH', '')}"
        print(f"Library search path updated: {lib_dir}")

else:
    print("CRITICAL ERROR: 'gmx' executable not found.")
    print("Please ensure 'MY_GROMACS_GPU.zip' is uploaded and contains the bin/gmx executable.")
    sys.exit()

# ==============================================================================
# 2. WORKSPACE PREPARATION
# ==============================================================================
print("\nPreparing simulation workspace...")

os.chdir("/content")

# Cleanup of previous CHARMM-GUI directories to prevent collision
for d in os.listdir("."):
    if os.path.isdir(d) and "charmm-gui" in d:
        shutil.rmtree(d)

# Input Archive Verification
tgz_files = glob.glob("*.tgz")
if not tgz_files:
    print("Error: Input archive (.tgz) not found. Upload 'charmm-gui.tgz'.")
    sys.exit()

# Extraction of most recent archive
latest_file = max(tgz_files, key=os.path.getctime)
subprocess.run(f"tar -xzvf '{latest_file}' > /dev/null", shell=True)

# Directory Navigation
# Locates the nested 'gromacs' directory generated by CHARMM-GUI
work_dir = None
for root, dirs, files in os.walk("."):
    if "gromacs" in dirs and "charmm-gui" in root:
        work_dir = os.path.join(root, "gromacs")
        break

if not work_dir: sys.exit("Error: Valid gromacs subdirectory not found.")
os.chdir(work_dir)

# Parameter File (Forcefield) Integration
# Copies toppar directory if missing from the working path
toppar_src = os.path.join(os.path.dirname(work_dir), "toppar")
if os.path.exists(toppar_src) and not os.path.exists("toppar"):
    shutil.copytree(toppar_src, "toppar")

# ==============================================================================
# 3. SIMULATION EXECUTION (GPU ACCELERATED)
# ==============================================================================
print(f"Working Directory: {work_dir}")
print("Starting MD execution sequence...")

# Alias for executable path
gmx = gmx_executable

try:
    # Phase 1: Energy Minimization (Steepest Descent)
    if not os.path.exists("step4.0_minimization.gro"):
        print("Executing Energy Minimization...")
        # Select topology file
        top = "topol.top" if os.path.exists("topol.top") else "step3_input.top"

        # Pre-processing (grompp)
        subprocess.run(f"'{gmx}' grompp -f step4.0_minimization.mdp -o step4.0_minimization.tpr -c step3_input.gro -r step3_input.gro -p {top} -maxwarn 2", shell=True, check=True)
        # Execution (mdrun)
        subprocess.run(f"'{gmx}' mdrun -v -deffnm step4.0_minimization", shell=True, check=True)

    # Phase 2: NVT/NPT Equilibration
    print("\nExecuting Equilibration...")
    if not os.path.exists("step4.1_equilibration.gro"):
        top = "topol.top" if os.path.exists("topol.top") else "step3_input.top"
        # Pre-processing
        subprocess.run(f"'{gmx}' grompp -f step4.1_equilibration.mdp -o step4.1_equilibration.tpr -c step4.0_minimization.gro -r step3_input.gro -p {top} -n index.ndx -maxwarn 2", shell=True, check=True)

        # Execution with GPU offload enabled
        print("Offloading calculation to GPU...")
        subprocess.run(f"'{gmx}' mdrun -v -deffnm step4.1_equilibration -nb gpu", shell=True, check=True)
        print("\nSimulation sequence complete.")

except subprocess.CalledProcessError as e:
    print(f"Execution Error: {e}")

In [ ]:
# @title Molecular Dynamics Production and Trajectory Analysis Pipeline
import os
import subprocess
import shutil
from google.colab import files

print("Initiating Production Dynamics Protocol...")

# ==============================================================================
# 1. EXECUTABLE RESOLUTION
# ==============================================================================
# verify binary path to ensure consistency with previous compilation steps
try:
    gmx_executable = subprocess.check_output("find /content -name 'gmx' -type f | head -n 1", shell=True).decode().strip()
    gmx_executable = os.path.abspath(gmx_executable)
    print(f"Engine verification successful: {gmx_executable}")
except:
    print("CRITICAL ERROR: GROMACS binary not located.")
    # Fallback to system default
    gmx_executable = "gmx"

# ==============================================================================
# 2. PRODUCTION DYNAMICS (NPT ENSEMBLE)
# ==============================================================================
# Directory context verification
if not os.path.exists("step4.1_equilibration.gro"):
    # Recursive search for working directory
    for root, dirs, f in os.walk("/content"):
        if "gromacs" in dirs:
            os.chdir(os.path.join(root, "gromacs"))
            break

try:
    if not os.path.exists("step5_production.gro"):
        print("\nExecuting Production Run (1ns)...")
        print("(Generating trajectory data)")

        # Assembler (grompp): Combines topology, coordinates, and control parameters
        # Quotes added to executable path to handle potential whitespace in filenames
        cmd_grompp = f"'{gmx_executable}' grompp -f step5_production.mdp -o step5_production.tpr -c step4.1_equilibration.gro -t step4.1_equilibration.cpt -p topol.top -n index.ndx -maxwarn 2"
        subprocess.run(cmd_grompp, shell=True, check=True)

        # Integrator (mdrun): GPU-accelerated execution
        cmd_mdrun = f"'{gmx_executable}' mdrun -v -deffnm step5_production -nb gpu"
        subprocess.run(cmd_mdrun, shell=True, check=True)
        print("Production simulation complete.")
    else:
        print("(Existing production data detected. Skipping execution.)")

    # ==============================================================================
    # 3. TRAJECTORY POST-PROCESSING & ARCHIVAL
    # ==============================================================================
    print("\nProcessing trajectory (Solvent removal and PBC correction)...")

    # trjconv: Remove water and center protein to facilitate visualization
    # Input '1 1' selects Protein group for centering and output
    cmd_trj = f"echo 1 1 | '{gmx_executable}' trjconv -s step5_production.tpr -f step5_production.xtc -o final_movie_no_water.xtc -pbc mol -center"
    subprocess.run(cmd_trj, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    print("\nPackaging simulation artifacts...")
    files_to_pack = [
        "final_movie_no_water.xtc",  # Post-processed Trajectory (Dry)
        "step5_production.xtc",      # Raw Trajectory (With Solvent)
        "step5_production.gro",      # Final Coordinate Frame
        "step3_input.pdb",           # Reference Structure
        "step5_production.log",      # Simulation Log
        "topol.top"                  # System Topology
    ]

    # Staging directory for export
    if os.path.exists("DOWNLOAD_ME"): shutil.rmtree("DOWNLOAD_ME")
    os.makedirs("DOWNLOAD_ME")

    for f in files_to_pack:
        if os.path.exists(f):
            shutil.copy(f, "DOWNLOAD_ME")

    # Compression and Export
    shutil.make_archive("FINAL_SIMULATION_DATA", 'zip', "DOWNLOAD_ME")
    print("\nPipeline Complete. Archive generated: 'FINAL_SIMULATION_DATA.zip'")

    # Trigger browser download
    files.download("FINAL_SIMULATION_DATA.zip")

except subprocess.CalledProcessError as e:
    print(f"Execution Failure: {e}")

In [ ]:
# @title Simulation Data Archival and Retrieval System
import os
import shutil
from google.colab import files

print("Initiating data packaging and archival sequence...")

# 1. ARTIFACT DEFINITION
# List of critical output files required for post-hoc analysis
target_files = [
    "final_movie_no_water.xtc",  # Post-processed trajectory (Solvent removed)
    "step5_production.xtc",      # Raw trajectory (Full system)
    "step5_production.gro",      # Final coordinate state
    "step5_production.log",      # Execution log
    "step5_production.edr",      # Energy data (Virial, Potential, Kinetic)
    "step3_input.pdb",           # Reference structure (Topology mapping)
    "topol.top"                  # System topology
]

# 2. STAGING ENVIRONMENT SETUP
output_folder = "FINAL_SIMULATION_DATA"
if os.path.exists(output_folder): shutil.rmtree(output_folder)
os.makedirs(output_folder)

# 3. RECURSIVE FILE SEARCH AND AGGREGATION
count = 0
# Scan entire content directory to locate artifacts regardless of working directory
for root, dirs, file_list in os.walk("/content"):
    for filename in target_files:
        if filename in file_list:
            # Construct absolute paths
            src = os.path.join(root, filename)
            dst = os.path.join(output_folder, filename)

            # Prevent duplicate copies if file found in multiple locations
            if not os.path.exists(dst):
                shutil.copy(src, dst)
                count += 1
                print(f"   - Archived: {filename}")

if count == 0:
    print("Error: No simulation artifacts located. Verify Step 5 completion.")
else:
    # 4. COMPRESSION AND EXPORT
    print(f"\nCompressing {count} artifacts...")
    shutil.make_archive(output_folder, 'zip', output_folder)

    print("Initiating download stream: 'FINAL_SIMULATION_DATA.zip'...")
    try:
        files.download(f"{output_folder}.zip")
        print("Download sequence started.")
    except Exception as e:
        print(f"Automatic download failed: {e}")
        print(f"Manual retrieval required: Locate '{output_folder}.zip' in file browser.")

In [ ]:
# @title GROMACS Binary Serialization and Export
import os
import shutil
from google.colab import files

print("Initiating binary serialization sequence...")

# Target directory containing the compiled CUDA-enabled binaries
install_dir = "/usr/local/gromacs"
output_filename = "MY_GROMACS_GPU"

# Verification of installation integrity
if os.path.exists(install_dir):
    # Compression: Archives the installation tree to preserve executable permissions
    shutil.make_archive(output_filename, 'zip', install_dir)
    print(f"Archive generated: {output_filename}.zip")

    # Artifact Transfer
    print("Initiating transfer...")
    try:
        files.download(f"{output_filename}.zip")
        print("Download started. Retain archive for rapid deployment in future sessions (bypasses compilation).")
    except Exception as e:
        print(f"Automatic transfer failed: {e}")
        print(f"Manual retrieval required: Locate '{output_filename}.zip' in file browser.")
else:
    print("Error: Installation directory not found. Verify compilation success.")

In [ ]:
# @title Recursive Directory Scan and Artifact Verification
import os

print("Initiating file system traversal...")

# State flags for integrity check
found_mdp = False
found_gro = False

# Walk the directory tree to map available inputs
for root, dirs, files in os.walk("."):
    # Exclude system configuration directories (Colab default)
    if ".config" in root:
        continue

    print(f"Scanning Directory: {root}")
    for f in files:
        # Filter for relevant simulation inputs, topologies, and coordinates
        if f.endswith((".tgz", ".pdb", ".gro", ".mdp", ".top", ".inp")):
            print(f"   - Detected: {f}")

        # Update flags upon detection of critical file types
        if f.endswith(".mdp"): found_mdp = True
        if f.endswith(".gro"): found_gro = True

print("-" * 30)
# Final Integrity Report
if not found_mdp:
    print("Error: GROMACS parameters (.mdp) not located.")
    print("Potential Cause: Input generation incomplete or directory path error.")
else:
    print("Verification Successful: Input parameters located.")

In [ ]:
# @title Real-Time Simulation Telemetry and Performance Monitor
import os
import subprocess

print("Acquiring runtime telemetry...")

# 1. LOG FILE IDENTIFICATION
log_filename = "step4.1_equilibration.log"
log_path = None

# Recursive directory traversal to locate active log
for root, dirs, files in os.walk("."):
    if log_filename in files:
        log_path = os.path.join(root, log_filename)
        break

if log_path:
    print(f"Log file located: {log_path}")
    print("-" * 50)

    # Read the final 20 lines to assess current state
    try:
        output = subprocess.check_output(f"tail -n 20 '{log_path}'", shell=True).decode()
        print(output)
    except subprocess.CalledProcessError:
        print("Error: Unable to read log stream.")

    print("-" * 50)

    # 2. HEURISTIC STATE ANALYSIS
    if "Performance:" in output:
        print("\nStatus: Execution Complete.")
        print("Action: Proceed to Production Phase.")
    elif "Step" in output:
        print("\nStatus: Active (Running).")
        print("Performance Assessment (ns/day):")
        print("   - High Throughput (>15 ns/day): GPU Acceleration Active.")
        print("   - Low Throughput (< 1 ns/day): CPU Fallback Detected (Inefficient).")
    else:
        print("\nStatus: Initialization/Pre-equilibration sequence.")

else:
    print("Error: Equilibration log not initialized.")

    # Fallback Diagnostic: Check if previous step (Minimization) succeeded
    min_log_found = False
    for root, dirs, files in os.walk("."):
        if "step4.0_minimization.log" in files:
            min_log_found = True
            break

    if min_log_found:
        print("Diagnostic: Minimization (Step 4.0) completed successfully.")
        print("Root Cause: Failure occurred during Equilibration (Step 4.1) startup.")
    else:
        print("Diagnostic: No simulation logs detected. Verify GROMACS execution.")

In [ ]:
# @title Reference Structure Post-Processing (Solvent Removal)
import os
import subprocess
from google.colab import files

print("Generating solvent-free reference coordinates...")

# Dynamic executable selection (Prioritize CUDA-enabled binary)
gmx = "gmx_gpu" if os.path.exists("/usr/bin/gmx_gpu") else "gmx"

# trjconv configuration:
# Purpose: Generate a dry PDB file to match the atom count of the processed trajectory (for VMD/PyMOL loading).
# -f: Input structure (Step 3 PDB)
# -s: Input topology (Step 5 TPR)
# -o: Output (Dry PDB)
# -center: Centers the protein in the box
# -pbc mol: Corrects periodic boundary conditions
# Input pipe '1 1': Selects 'Protein' group for both centering and output
cmd = f"echo 1 1 | {gmx} trjconv -f step3_input.pdb -s step5_production.tpr -o clean_structure.pdb -center -pbc mol"

try:
    subprocess.run(cmd, shell=True, check=True)
    print("File generated: clean_structure.pdb")

    print("Initiating download...")
    files.download("clean_structure.pdb")
except subprocess.CalledProcessError as e:
    print(f"Execution failed: {e}")